In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

train_df = pd.read_csv("../data/train_clean.csv")
test_df = pd.read_csv("../data/test_clean.csv")

full_df = pd.concat(
    [train_df, test_df],
    ignore_index=True
)

X = full_df["clean_poem"]
y = full_df["Genre"]

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

le = LabelEncoder()

y_train = le.fit_transform(y_train_text)
y_test = le.transform(y_test_text)

In [2]:
pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 598.5 kB/s eta 0:00:40
    --------------------------------------- 0.5/24.4 MB 598.5 kB/s eta 0:00:40
   - -------------------------------------- 0.8/24.4 MB 745.8 kB/s eta 0:00:32
   - -------------------------------------- 1.0/24.4 MB 774.0 kB/s eta 0:00:31
   - -------------------------------------- 1.0/24.4 MB 774.0 kB/s eta 0:00:31
   - -------------------------------------- 1.0/24.4 MB 774.0 kB/s eta 0:00:31
   -- ------------------------------------- 1.3/24.4 MB 684.9 kB/s eta 0:00:34
   -- ------------------------------------- 1.6/24.4 MB 687.7 kB/s eta 0:00:34
   -- ------------------------------------- 1.6/24.4 MB 687.7 kB/s eta 0:00:34
   -- ------------------------------------- 1.6/24.4 MB 687.7 kB/s eta 0:00:34


In [3]:
from gensim.models import Word2Vec

sentences = [
    text.split()
    for text in X_train_text
]

w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [4]:
import numpy as np

def document_vector(text):

    words = text.split()

    vectors = []

    for word in words:

        if word in w2v_model.wv:
            vectors.append(
                w2v_model.wv[word]
            )

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [5]:
X_train_w2v = np.array(
    [document_vector(text)
     for text in X_train_text]
)

X_test_w2v = np.array(
    [document_vector(text)
     for text in X_test_text]
)

print(X_train_w2v.shape)

(788, 100)


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr = LogisticRegression(max_iter=1000)

lr.fit(
    X_train_w2v,
    y_train
)

pred = lr.predict(
    X_test_w2v
)

print(
    accuracy_score(
        y_test,
        pred
    )
)

0.25380710659898476
